In [6]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

# 1. Configuração Inteligente de Caminhos
# Descobre onde o projeto está localizado
base_path = Path().absolute()
nome_ficheiro = 'g62_Banks_Cards.csv'

def encontrar_ficheiro(nome, caminho_base):
    # Procura o ficheiro na pasta atual e subpastas
    for caminho in caminho_base.rglob(nome):
        return caminho
    return None

csv_path = encontrar_ficheiro(nome_ficheiro, base_path)

try:
    if csv_path is None:
        raise FileNotFoundError(f"Não encontrei o ficheiro '{nome_ficheiro}' em nenhuma pasta de {base_path}")

    # 2. Leitura do CSV
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip() # Limpa espaços nos nomes das colunas
    print(f"✅ Ficheiro encontrado e lido em: {csv_path}")

    # 3. Configuração da Base de Dados
    # Vamos guardar na pasta 'data' dentro de onde o CSV foi encontrado
    db_dir = csv_path.parent / 'data'
    if not db_dir.exists():
        db_dir.mkdir(parents=True, exist_ok=True)
    
    db_path = db_dir / 'g62_bank_cards.db'
    conn = sqlite3.connect(str(db_path))

    # --- IMPORTAÇÃO (Usando replace para garantir que as colunas ficam certas) ---

    # Bank
    df[['bank_id', 'designation', 'founding_date']].drop_duplicates().to_sql('Bank', conn, if_exists='replace', index=False)
    
    # Branch
    if 'branch_id' in df.columns:
        df[['branch_id', 'branch_location', 'bank_id']].dropna(subset=['branch_id']).drop_duplicates().to_sql('Branch', conn, if_exists='replace', index=False)
    
    # Card
    df[['card_id', 'name', 'card_type']].drop_duplicates().to_sql('Card', conn, if_exists='replace', index=False)
    
    # Transaction
    df[['transaction_date', 'amount', 'card_id', 'branch_id']].to_sql('Transaction', conn, if_exists='replace', index=False)

    conn.commit()
    conn.close()
    print(f"🚀 SUCESSO! Dados importados para: {db_path}")

except Exception as e:
    print(f"❌ Erro: {e}")



✅ Ficheiro encontrado e lido em: c:\Users\teres\OneDrive\Documentos\GitHub\g62_project\data\g62_Banks_Cards.csv
🚀 SUCESSO! Dados importados para: c:\Users\teres\OneDrive\Documentos\GitHub\g62_project\data\data\g62_bank_cards.db
